In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv('users_commodities_india_geo.csv')
df['commodity_list'] = df['commodity'].apply(
    lambda x: [c.strip() for c in x.split(';')]
)
print(df.shape)
df.head()

(1000, 12)


,user_id,commodity,role,city,state,latitude_raw,longitude_raw,latitude_normalized,longitude_normalized,min_quantity_mt,max_quantity_mt,commodity_list
0,1,sugar;rice;cotton,trader,Dindigul,Tamil Nadu,10.3673,77.9757,-0.824644,-0.312021,63,285,"[sugar, rice, cotton]"
1,2,rice,trader,Mumbai,Maharashtra,19.0760,72.8777,-0.179556,-0.663607,61,354,[rice]
2,3,cotton;rice,exporter,Indore,Madhya Pradesh,22.7196,75.8577,0.090341,-0.458090,55,203,"[cotton, rice]"
3,4,rice;sugar,trader,Gandhinagar,Gujarat,23.2156,72.6369,0.127081,-0.680214,99,380,"[rice, sugar]"
4,5,cotton;sugar,exporter,Ludhiana,Punjab,30.9010,75.8573,0.696370,-0.458117,73,188,"[cotton, sugar]"


In [2]:
ALL_COMMODITIES = sorted(set(
    c for lst in df['commodity_list'] for c in lst
))
print("Commodities found:", ALL_COMMODITIES)

COMMODITY_BOOST = 0.9

def encode_commodity(commodity_list):
    vec = np.zeros(len(ALL_COMMODITIES))
    for c in commodity_list:
        if c in ALL_COMMODITIES:
            vec[ALL_COMMODITIES.index(c)] = 1.0
    return vec * COMMODITY_BOOST

commodity_vecs = np.array([
    encode_commodity(row['commodity_list'])
    for _, row in df.iterrows()
])

print("Dimension names:", [f"has_{c}" for c in ALL_COMMODITIES])
print()
for i in range(5):
    row = df.iloc[i]
    raw = commodity_vecs[i] / COMMODITY_BOOST
    vals = dict(zip([f"has_{c}" for c in ALL_COMMODITIES], raw))
    print(f"User {row['user_id']} | {row['commodity']:20s} → {vals}")

Commodities found: ['cotton', 'rice', 'sugar']
Dimension names: ['has_cotton', 'has_rice', 'has_sugar']

User 1 | sugar;rice;cotton    → {'has_cotton': np.float64(1.0), 'has_rice': np.float64(1.0), 'has_sugar': np.float64(1.0)}
User 2 | rice                 → {'has_cotton': np.float64(0.0), 'has_rice': np.float64(1.0), 'has_sugar': np.float64(0.0)}
User 3 | cotton;rice          → {'has_cotton': np.float64(1.0), 'has_rice': np.float64(1.0), 'has_sugar': np.float64(0.0)}
User 4 | rice;sugar           → {'has_cotton': np.float64(0.0), 'has_rice': np.float64(1.0), 'has_sugar': np.float64(1.0)}
User 5 | cotton;sugar         → {'has_cotton': np.float64(1.0), 'has_rice': np.float64(0.0), 'has_sugar': np.float64(1.0)}


In [3]:
ROLE_AFFINITY = {
    'trader':   {'want_broker': 0.55, 'want_exporter': 0.3, 'want_trader': 0.2},
    'broker':   {'want_broker': 0.33,'want_exporter': 0.33,'want_trader': 0.33},
    'exporter': {'want_broker': 0.7, 'want_exporter': 0.2, 'want_trader': 0.3},
}

# What each role OFFERS — used for candidate vectors
ROLE_OFFERS = {
    'trader':   {'is_broker': 0.0, 'is_exporter': 0.0, 'is_trader': 1.0},
    'broker':   {'is_broker': 1.0, 'is_exporter': 0.0, 'is_trader': 0.0},
    'exporter': {'is_broker': 0.0, 'is_exporter': 1.0, 'is_trader': 0.0},
}

ROLE_DIMS  = ['want_broker', 'want_exporter', 'want_trader']
ROLE_BOOST = 1.5

def encode_role_searcher(role):
    """What this user WANTS — used for the query vector."""
    affinity = ROLE_AFFINITY[role]
    return np.array([affinity[d] for d in ROLE_DIMS]) * ROLE_BOOST

def encode_role_candidate(role):
    offers = ROLE_OFFERS[role]
    return np.array([
        offers['is_broker'],
        offers['is_exporter'],
        offers['is_trader']
    ]) * ROLE_BOOST

# Stored vectors use candidate encoding (what they ARE)
role_vecs = np.array([
    encode_role_candidate(row['role'])
    for _, row in df.iterrows()
])

print("Dimension names:", ROLE_DIMS)
print()
print("Candidate vectors (what they ARE — stored in DB):")
for role in ['trader', 'broker', 'exporter']:
    v = encode_role_candidate(role) / ROLE_BOOST
    print(f"  {role:10s} → {dict(zip(ROLE_DIMS, v))}")

print()
print("Searcher vectors (what they WANT — used at query time only):")
for role in ['trader', 'broker', 'exporter']:
    v = encode_role_searcher(role) / ROLE_BOOST
    print(f"  {role:10s} → {dict(zip(ROLE_DIMS, v))}")

print()
print("Dot product preview for trader searching:")
trader_want = encode_role_searcher('trader')
for role in ['broker', 'exporter', 'trader']:
    candidate_offer = encode_role_candidate(role)
    dot = np.dot(trader_want, candidate_offer)
    print(f"  trader → {role}: dot = {dot:.3f}")

Dimension names: ['want_broker', 'want_exporter', 'want_trader']

Candidate vectors (what they ARE — stored in DB):
  trader     → {'want_broker': np.float64(0.0), 'want_exporter': np.float64(0.0), 'want_trader': np.float64(1.0)}
  broker     → {'want_broker': np.float64(1.0), 'want_exporter': np.float64(0.0), 'want_trader': np.float64(0.0)}
  exporter   → {'want_broker': np.float64(0.0), 'want_exporter': np.float64(1.0), 'want_trader': np.float64(0.0)}

Searcher vectors (what they WANT — used at query time only):
  trader     → {'want_broker': np.float64(0.55), 'want_exporter': np.float64(0.3), 'want_trader': np.float64(0.20000000000000004)}
  broker     → {'want_broker': np.float64(0.33), 'want_exporter': np.float64(0.33), 'want_trader': np.float64(0.33)}
  exporter   → {'want_broker': np.float64(0.6999999999999998), 'want_exporter': np.float64(0.20000000000000004), 'want_trader': np.float64(0.3)}

Dot product preview for trader searching:
  trader → broker: dot = 1.238
  trader → ex

In [27]:
# city_enc  = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
# state_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# city_vecs  = city_enc.fit_transform(df[['city']])
# state_vecs = state_enc.fit_transform(df[['state']])

# print("City dimensions:",  city_enc.categories_[0].tolist())
# print("State dimensions:", state_enc.categories_[0].tolist())
# print()
# print("City vec shape:", city_vecs.shape)
# print("State vec shape:", state_vecs.shape)

import numpy as np

def encode_geo_3d(lat, lon, boost=1.0):
    """
    Normalizes raw Lat/Lon into 3D Cartesian coordinates.
    Works perfectly with Cosine Similarity.
    """
    # 1. Convert degrees to radians
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    
    # 2. Project to 3D Space (X, Y, Z)
    # This places the point on a unit sphere (radius 1)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    
    # 3. Apply weight/boost
    return np.array([x, y, z]) * boost

GEO_BOOST = 3.0 # Increase this if you want distance to be more important

geo_vec_list = []
for idx, row in df.iterrows():
    vec = encode_geo_3d(row['latitude_raw'], row['longitude_raw'], GEO_BOOST)
    geo_vec_list.append(vec)

geo_vecs = np.array(geo_vec_list)

In [ ]:
# Final vector layout:
# [ has_cotton | has_rice | has_sugar | want_broker | want_exporter | want_trader | city_onehot... | state_onehot... ]

master_vectors = np.hstack([
    commodity_vecs,   # explicitly named has_X dims
    role_vecs,        # explicitly named want_X dims
    geo_vecs,
    quant_vecs
])

df['vector'] = master_vectors.tolist()

# Print dimension layout so you know exactly what each position means
n_comm  = commodity_vecs.shape[1]
n_role  = role_vecs.shape[1]
# n_city  = city_vecs.shape[1]
# n_state = state_vecs.shape[1]
n_geo   = geo_vecs.shape[1]
total   = master_vectors.shape[1]

print(f"Total vector dims: {total}")
print(f"  [0 : {n_comm}]              → commodity  ({[f'has_{c}' for c in ALL_COMMODITIES]})")
print(f"  [{n_comm} : {n_comm+n_role}]          → role       ({ROLE_DIMS})")
print(f"  [{n_comm+n_role} : {total}]  → geo        ({n_geo} dims)")
# print(f"  [{n_comm+n_role} : {n_comm+n_role}]   → city       ({city_enc.categories_[0].tolist()})")
# print(f"  [{n_comm+n_role} : {total}]  → state      ({state_enc.categories_[0].tolist()})")

Total vector dims: 9
  [0 : 3]              → commodity  (['has_cotton', 'has_rice', 'has_sugar'])
  [3 : 6]          → role       (['want_broker', 'want_exporter', 'want_trader'])
  [6 : 9]  → geo        (3 dims)


In [29]:
# Quantity stored as raw metadata only — used for DB filter, not in vector
vector_db = []

for idx, row in df.iterrows():
    vector_db.append({
        "id":     int(row['user_id']),
        "vector": master_vectors[idx],
        "meta": {
            "user_id":        int(row['user_id']),
            "role":           row['role'],
            "commodity":      row['commodity'],
            "commodity_list": row['commodity_list'],
            "city":           row['city'],
            "state":          row['state'],
            'latitude_raw': row['latitude_raw'],    # Add this line
            'longitude_raw': row['longitude_raw'],
            "qty_min":        int(row['min_quantity_mt']),
            "qty_max":        int(row['max_quantity_mt']),
        }
    })

print(f"Vector DB: {len(vector_db)} records")
print("\nSample:")
r = vector_db[0]
print(f"  id: {r['id']}")
print(f"  meta: {r['meta']}")
print(f"  vector dims: {len(r['vector'])}")
print(f"  vector (commodity+role section): {r['vector'][:n_comm+n_role]}")
print(f"  names: {[f'has_{c}' for c in ALL_COMMODITIES] + ROLE_DIMS}")

Vector DB: 1000 records

Sample:
  id: 1
  meta: {'user_id': 1, 'role': 'trader', 'commodity': 'sugar;rice;cotton', 'commodity_list': ['sugar', 'rice', 'cotton'], 'city': 'Dindigul', 'state': 'Tamil Nadu', 'latitude_raw': 10.3673, 'longitude_raw': 77.9757, 'qty_min': 63, 'qty_max': 285}
  vector dims: 9
  vector (commodity+role section): [0.9 0.9 0.9 0.  0.  1.5]
  names: ['has_cotton', 'has_rice', 'has_sugar', 'want_broker', 'want_exporter', 'want_trader']


In [30]:
def search(searcher_id, top_k=50):
    # 1. Locate the searcher in the local DB
    searcher = next(r for r in vector_db if r['meta']['user_id'] == searcher_id)
    searcher_meta = searcher['meta']

    # 2. Build query vector components
    s_commodity = encode_commodity(searcher_meta['commodity_list'])
    s_role      = encode_role_searcher(searcher_meta['role'])
    
    # --- UPDATE: Replace state_enc with 3D Geo Normalization ---
    s_geo = encode_geo_3d(
        searcher_meta['latitude_raw'], 
        searcher_meta['longitude_raw'], 
        boost=GEO_BOOST
    )

    # Combine parts (Make sure the order matches how you built master_vectors)
    s_vec = np.concatenate([s_commodity, s_role, s_geo]).reshape(1, -1)
    # -----------------------------------------------------------

    # 3. All users except searcher
    candidates = [
        r for r in vector_db
        if r['meta']['user_id'] != searcher_id
    ]

    if not candidates:
        print("No candidates found.")
        return pd.DataFrame()

    # 4. Calculate similarity
    cand_vecs = np.array([r['vector'] for r in candidates])
    sims = cosine_similarity(s_vec, cand_vecs)[0]

    # 5. Sort and prepare results
    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for i in top_idx:
        m = candidates[i]['meta']
        results.append({
            'user_id':    m['user_id'],
            'role':       m['role'],
            'commodity':  m['commodity'],
            'city':       m['city'],
            'state':      m['state'],
            # Optional: Add coordinates to result for verification
            'lat':        m.get('latitude_raw'),
            'lon':        m.get('longitude_raw'),
            'qty_range':  f"{m['qty_min']}–{m['qty_max']}mt",
            'similarity': round(float(sims[i]), 4),
        })

    return pd.DataFrame(results)

In [31]:
TEST_USER_ID = 68

s = next(r['meta'] for r in vector_db if r['meta']['user_id'] == TEST_USER_ID)
print(f"Searcher: User {s['user_id']} | {s['role']} | {s['commodity']} | {s['state']} |{s['latitude_raw']}, {s['longitude_raw']} | {s['qty_min']}–{s['qty_max']}mt")
print("=" * 70)

results = search(TEST_USER_ID, top_k=15)
print(results.to_string(index=False))

print("\nWhat to verify:")
print("1. Same commodity users rank highest")
print("2. For a trader: brokers outrank exporters, exporters outrank traders")
print("3. Same city users rank above different city (same commodity+role)")
print("4. Anyone with zero qty overlap is absent from results entirely")

Searcher: User 68 | trader | sugar;rice | Tamil Nadu |11.6643, 78.146 | 54–229mt
 user_id   role  commodity       city       state     lat     lon qty_range  similarity
     278 broker rice;sugar      Salem  Tamil Nadu 11.6643 78.1460  58–179mt      0.9707
     880 broker sugar;rice   Dindigul  Tamil Nadu 10.3673 77.9757  23–155mt      0.9705
     651 broker rice;sugar   Dindigul  Tamil Nadu 10.3673 77.9757  49–168mt      0.9705
     414 broker rice;sugar   Dindigul  Tamil Nadu 10.3673 77.9757  67–302mt      0.9705
     314 broker rice;sugar  Bengaluru   Karnataka 12.9716 77.5946  40–255mt      0.9705
     700 broker sugar;rice     Mysuru   Karnataka 12.2958 76.6394  27–304mt      0.9705
     149 broker rice;sugar    Vellore  Tamil Nadu 12.9689 79.1288  34–127mt      0.9704
     874 broker rice;sugar  Mangaluru   Karnataka 12.8656 74.8547  80–327mt      0.9694
     649 broker sugar;rice      Hubli   Karnataka 15.3647 75.1240  20–195mt      0.9682
     221 broker sugar;rice   Belagavi  

## Chroma DB for testing 
 

In [32]:
import chromadb

# PersistentClient writes to disk — data survives kernel restarts.
# Change the path to wherever you want the DB files stored.
chroma_client = chromadb.PersistentClient(path="./chroma_commodity_db")

COLLECTION_NAME = 'commodity_users'

# Delete and recreate so re-runs are idempotent
existing = [c.name for c in chroma_client.list_collections()]
if COLLECTION_NAME in existing:
    chroma_client.delete_collection(COLLECTION_NAME)

# hnsw:space='cosine' matches your existing cosine_similarity logic.
# ChromaDB stores cosine *distance* = 1 - similarity, so highest sim = lowest dist.
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

print(f'Collection "{COLLECTION_NAME}" ready.')


RuntimeError: Chroma is running in http-only client mode, and can only be run with 'chromadb.api.fastapi.FastAPI' or 'chromadb.api.async_fastapi.AsyncFastAPI' as the chroma_api_impl.             see https://docs.trychroma.com/guides#using-the-python-http-only-client for more information.

In [ ]:
# Upsert all vectors into ChromaDB.
#
# ChromaDB requires:
#   ids        -> list[str]           (must be strings)
#   embeddings -> list[list[float]]   (your master_vectors)
#   metadatas  -> list[dict]          (flat key-value; no nested lists)

BATCH_SIZE = 100

ids, embeddings, metadatas = [], [], []

for record in vector_db:
    m = record['meta']
    ids.append(str(record['id']))
    embeddings.append([float(v) for v in record['vector']])
    metadatas.append({
        'user_id':        m['user_id'],
        'role':           m['role'],
        'commodity':      m['commodity'],
        # ChromaDB metadata cannot store Python lists — join to a semicolon string
        'commodity_list': ';'.join(m['commodity_list']),
        
        'city':           m['city'],
        'state':          m['state'],
        'qty_min':        m['qty_min'],
        'qty_max':        m['qty_max'],
    })

for start in range(0, len(ids), BATCH_SIZE):
    end = start + BATCH_SIZE
    collection.upsert(
        ids=ids[start:end],
        embeddings=embeddings[start:end],
        metadatas=metadatas[start:end],
    )

print(f'Upserted {collection.count()} records into ChromaDB.')
print(f'Vector dimension: {len(embeddings[0])}')


Upserted 1000 records into ChromaDB.
Vector dimension: 16


In [ ]:
def search_chroma(searcher_id, top_k=15, filter_state=True, filter_role=None):
    """
    ChromaDB-backed search. Query vector uses WANT encoding (same as original search()).
    ChromaDB handles the ANN search internally via HNSW.

    Parameters
    ----------
    searcher_id  : int  -- user_id of the searcher
    top_k        : int  -- number of results to return
    filter_state : bool -- if True, restrict results to same state as searcher
    filter_role  : str  -- if given (e.g. 'broker'), restrict candidates to that role
    """
    # 1. Pull searcher metadata from ChromaDB
    result = collection.get(ids=[str(searcher_id)], include=['metadatas'])
    if not result['ids']:
        print(f'User {searcher_id} not found in ChromaDB.')
        return pd.DataFrame()

    s_meta = result['metadatas'][0]
    s_role = s_meta['role']
    s_commodities = [c.strip() for c in s_meta['commodity_list'].split(';')]

    # 2. Build query vector with WANT encoding (not stored in DB, only used at query time)
    s_commodity = encode_commodity(s_commodities)
    s_role_vec  = encode_role_searcher(s_role)
    s_state_vec = state_enc.transform([[s_meta['state']]])[0]
    query_vec   = np.concatenate([s_commodity, s_role_vec, s_state_vec]).tolist()

    # 3. Build metadata filter — exclude self, optionally narrow by state / role
    conditions = [{'user_id': {'$ne': searcher_id}}]
    if filter_state:
        conditions.append({'state': {'$eq': s_meta['state']}})
    if filter_role:
        conditions.append({'role': {'$eq': filter_role}})

    where_clause = conditions[0] if len(conditions) == 1 else {'$and': conditions}

    # 4. Query ChromaDB
    query_result = collection.query(
        query_embeddings=[query_vec],
        n_results=top_k,
        where=where_clause,
        include=['metadatas', 'distances'],
    )

    # 5. Format results.
    # ChromaDB returns cosine *distance* (0=identical, 2=opposite).
    # Convert back: similarity = 1 - distance  (matches your original 0-1 scale)
    rows = []
    for meta, dist in zip(query_result['metadatas'][0], query_result['distances'][0]):
        rows.append({
            'user_id':   meta['user_id'],
            'role':      meta['role'],
            'commodity': meta['commodity'],
            'city':      meta['city'],
            'state':     meta['state'],
            'qty_range': f"{meta['qty_min']}–{meta['qty_max']}mt",
            'similarity': round(1 - dist, 4),
        })

    return pd.DataFrame(rows)


In [ ]:
# Test ChromaDB search and verify rankings match the original in-memory search
TEST_USER_ID = 53

s_meta_raw = next(r['meta'] for r in vector_db if r['meta']['user_id'] == TEST_USER_ID)
print(f"Searcher: User {TEST_USER_ID} | {s_meta_raw['role']} | "
      f"{s_meta_raw['commodity']} | {s_meta_raw['state']} | "
      f"{s_meta_raw['qty_min']}–{s_meta_raw['qty_max']}mt")
print('=' * 70)

print('\n[ChromaDB Results]')
chroma_results = search_chroma(TEST_USER_ID, top_k=15, filter_state=False)
print(chroma_results.to_string(index=False))

print('\n[Original In-Memory Results]')
original_results = search(TEST_USER_ID, top_k=15)
print(original_results.to_string(index=False))

print('\n[Verification]')
match = chroma_results['user_id'].tolist() == original_results['user_id'].tolist()
print(f'Rankings match original: {match}')


Searcher: User 53 | trader | sugar | Tamil Nadu | 32–225mt

[ChromaDB Results]
 user_id     role         commodity            city      state qty_range  similarity
      71   broker      cotton;sugar         Chennai Tamil Nadu  41–204mt      0.8278
      92   broker      sugar;cotton        Dindigul Tamil Nadu  59–278mt      0.8278
     212   broker      cotton;sugar           Erode Tamil Nadu   32–96mt      0.8278
     595   broker      sugar;cotton        Dindigul Tamil Nadu  95–366mt      0.8278
     610   broker        rice;sugar     Thoothukudi Tamil Nadu  32–286mt      0.8278
     835   broker      cotton;sugar         Madurai Tamil Nadu  80–336mt      0.8278
     905   broker        sugar;rice Tiruchirappalli Tamil Nadu  21–190mt      0.8278
     337   broker rice;sugar;cotton      Coimbatore Tamil Nadu  49–157mt      0.7665
     362   broker sugar;cotton;rice         Madurai Tamil Nadu  21–171mt      0.7665
     375   broker rice;cotton;sugar         Chennai Tamil Nadu  30–120m

c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [ ]:
# Example: filter by state AND role at query time
# This is one of the key advantages over the in-memory approach
print('Brokers only, same state as searcher:')
filtered = search_chroma(TEST_USER_ID, top_k=10, filter_state=True, filter_role='broker')
print(filtered.to_string(index=False))


Brokers only, same state as searcher:
 user_id   role         commodity            city      state qty_range  similarity
      71 broker      cotton;sugar         Chennai Tamil Nadu  41–204mt      0.8278
      92 broker      sugar;cotton        Dindigul Tamil Nadu  59–278mt      0.8278
     212 broker      cotton;sugar           Erode Tamil Nadu   32–96mt      0.8278
     595 broker      sugar;cotton        Dindigul Tamil Nadu  95–366mt      0.8278
     610 broker        rice;sugar     Thoothukudi Tamil Nadu  32–286mt      0.8278
     835 broker      cotton;sugar         Madurai Tamil Nadu  80–336mt      0.8278
     905 broker        sugar;rice Tiruchirappalli Tamil Nadu  21–190mt      0.8278
     362 broker sugar;cotton;rice         Madurai Tamil Nadu  21–171mt      0.7665
     375 broker rice;cotton;sugar         Chennai Tamil Nadu  30–120mt      0.7665
     672 broker rice;sugar;cotton     Thoothukudi Tamil Nadu  45–309mt      0.7665


c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
